# Movie genre classifier

##### This notebook contains an implementation of my naive bayes text classifier to classify several movies into their genres based upon a plot description.

___

In [37]:
import pandas as pd

In [49]:
movies_df = pd.read_csv('data\wiki_movie_plots_deduped.csv')
movies_df.head()

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",American,Unknown,NaN,unknown,"https://en.wikipedia.org/wiki/Terrible_Teddy,_...",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,American,"George S. Fleming, Edwin S. Porter",NaN,unknown,https://en.wikipedia.org/wiki/Jack_and_the_Bea...,The earliest known adaptation of the classic f...


In [75]:
genre_description = (
    movies_df[['Genre', 'Plot']]
    .assign(Genre=movies_df['Genre'].str.split(r'[/,\-]'))
    .explode('Genre')
    .reset_index(drop=True)
)
genre_description.head(20)

,Genre,Plot
0,unknown,"A bartender is working at a saloon, serving dr..."
1,unknown,"The moon, painted with a smiling face hangs ov..."
2,unknown,"The film, just over a minute long, is composed..."
3,unknown,Lasting just 61 seconds and consisting of two ...
4,unknown,The earliest known adaptation of the classic f...
5,unknown,"Alice follows a large white rabbit down a ""Rab..."
6,western,The film opens with two bandits breaking into ...
7,comedy,The film is about a family who move to the sub...
8,unknown,The opening scene shows the interior of the ro...
9,unknown,Scenes are introduced using lines of the poem....


In [71]:
genres = genre_description['Genre'].unique()
print(genres)

['western' 'comedy' 'short' 'action' 'crime' 'film' 'biographical' 'drama'
 'adventure' 'fantasy' 'silent' 'sports' 'horror' 'drama,' 'historical'
 'documentary' 'serial' 'epic' 'comedy,' 'biography' 'comedy–drama'
 'romantic' 'mystery' 'romance' 'sexual' 'hygiene' 'exploitation' 'war'
 'spy' 'propaganda' 'ww1' 'biopic' 'animated' 'series' 'melodrama'
 'period' 'swashbuckler' 'fantasy,' 'family' 'thriller' 'dramatic'
 'mystery,' 'american' 'football' 'horror,' 'semi-staged' 'biblical'
 'race' 'romance,' 'crime,' 'musical' 'operetta' 'detective' 'musical,'
 'costume' 'prison' 'science' 'fiction' 'noir' 'animation' 'sci-fi'
 'adventure,' 'fiction,' 'sci-fi,' 'western,' 'sport' 'murder' '2-reeler'
 'comedy-drama' 'action,' 'sport,' 'animated,' 'military' 'music'
 'suspense' 'gangster' 'bio-pic' 'screwball' 'charlie' 'chan' 'cartoon'
 'aviation' 'war,' 'docudrama' 'comedy;' '6' 'separate' 'stories'
 'anthology' 'musical–comedy' 'biography,' 'subject' 'family,'
 'experimental' 'nature' 'edu

In [82]:
final_genres = [
    'action', 'adventure', 'animation', 'biographical', 'comedy', 'crime',
    'documentary', 'drama', 'family', 'fantasy', 'historical', 'horror',
    'musical', 'mystery', 'romance', 'sci-fi', 'sports', 'thriller', 'war',
    'western', 'romantic comedy', 'action-comedy', 'crime-drama', 'drama-romance',
    'horror-comedy', 'fantasy-adventure', 'superhero', 'animated-adventure',
    'musical-comedy', 'spy', 'period', 'biopic', 'political', 'psychological thriller',
    'science-fantasy', 'teen', 'family-comedy', 'documentary-biographical', 'experimental'
]
genre_map = {
    'sci fi': 'sci-fi',
    'sciencefiction': 'sci-fi',
    'science-fiction': 'sci-fi',
    'romcom': 'romantic comedy',
    'romantic-comedy': 'romantic comedy',
    'comedy-drama': 'drama-comedy',
    'action adventure': 'action-adventure',
    'fantasy comedy': 'fantasy-adventure',
    'musical comedy': 'musical-comedy',
    'animated adventure': 'animated-adventure',
    'biopic': 'biographical',
}

genre_description['Genre'] = genre_description['Genre'].str.strip().str.lower()
genre_description = genre_description[
    (genre_description['Genre'] != '') &
    (genre_description['Genre'] != 'unknown')
]
genre_description['Genre'] = genre_description['Genre'].replace(genre_map)
genre_description = genre_description[
    genre_description['Genre'].isin(final_genres)
].reset_index(drop=True)
genre_description.head()

,Genre,Plot
0,western,The film opens with two bandits breaking into ...
1,comedy,The film is about a family who move to the sub...
2,biographical,Boone's daughter befriends an Indian maiden as...
3,comedy,Before heading out to a baseball game at a nea...
4,comedy,The plot is that of a black woman going to the...


In [40]:
import nltk
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jack0\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [41]:
from nltk.corpus import stopwords

stopwords = set(stopwords.words('english'))
print(stopwords)

{'further', 'were', 'won', "they'd", 'then', 'up', 'it', 'now', 'on', 'ain', "it'd", 'or', "we're", 'its', 'needn', "didn't", "couldn't", 'just', "wouldn't", "you'd", 'all', 'do', "they'll", "it'll", 'this', "shan't", 'in', 'these', "weren't", 'we', 'itself', 'such', 've', 'above', 'over', 'most', "mightn't", 'himself', "he's", "they've", 'what', "won't", 'haven', 'yourselves', 'more', 'out', 'she', 'the', 'not', 'too', 'has', 'some', 'because', 'when', "you're", 'did', 'being', 'than', 'until', 'mightn', 'against', 'under', 'before', 'same', 'wouldn', "you'll", 'about', 'both', 'once', 'how', 'm', 'don', "she'll", 'between', 'why', "shouldn't", 'again', 'will', 'having', 'been', 'very', "hadn't", "he'll", 'didn', 'her', 'mustn', 'an', 'which', 'only', 'your', 'o', 'into', 'them', 'at', 'couldn', 'but', "it's", 'i', 'y', 'his', "should've", 'each', 'hadn', "hasn't", "he'd", 'myself', 'their', 'doesn', 'if', "doesn't", 'as', 'me', 'below', 'nor', 'by', 'had', 'weren', 'can', "i've", "ha

In [59]:
stopwords.add('film')
stopwords.add('features')

In [42]:
def clean_text(text):
    #Lowercase
    text = text.lower()
    #Split up words
    words = text.split()
    #Remove stopwords
    words = [w for w in words if w not in stopwords]
    #Remove punctuation
    words = [w for w in words if w.isalpha()]
    #Put string back together
    string = ' '.join(words)

    return string

In [65]:
processed = genre_description.copy()
processed['Plot'] = processed['Plot'].apply(clean_text)
processed.head()

,Genre,Plot
6,western,opens two bandits breaking railroad telegraph ...
7,comedy,family move hoping quiet things start go wife ...
10,short,rarebit fiend gorges welsh rarebit begins get ...
11,short,train traveling rockies hold created two thugs...
12,action,train traveling rockies hold created two thugs...


In [83]:
from bayes_classifier import TextClassifier

In [ ]:
genre_classifier = TextClassifier(processed)

comedy
